# Adult Sentence-Influence Matrix on Llama-3.1-8B

**Question:** in adult written prose and prepared spoken corpora, do **opening sentences carry outsized influence beyond what distance alone predicts** — replicating the anchor effect we found in child narrative production?

**Method:** for each document, segment into sentences and compute the **leave-one-sentence-out influence matrix**:

- For each target sentence `t` (with `t ≥ 2`), compute `ppl_full` (target predicted from all prior sentences `0..t-1` in order).
- For each prior sentence `i < t`, compute `ppl_dropped[i, t]` (target predicted from prior with sentence `i` removed).
- `influence[i, t] = log(ppl_dropped[i, t] / ppl_full)`. Positive = removing source `i` hurt target `t`.

**Critical contrast:** within each distance bin `d = t - i`, compare influence when `i = 0` (document opening) vs. all other source positions at the same distance. A pure-positional model predicts no difference. A discourse-anchor model predicts the opening contributes more.

**Corpora:** `gutenberg_fiction_en`, `ted_transcripts_en`. Adult discourse with well-formed sentence boundaries.

**Pilot N:** 15 docs/corpus × up to 25 sentences/doc = ~625 ablations × 30 docs ≈ 19k forward passes. ~15 min on H100 fp16.

**Output:** `My Drive/LRTIA/Results/sentence_influence_matrix/llama/<corpus_id>.json`. Per-doc records:
```
{ doc_id, n_sentences, sentences, sentence_token_lengths,
  influence_matrix: [[v_ij or null]],   # log(ppl_dropped/ppl_full); null = not computed
  ppl_full_per_target: [...] }
```

In [ ]:
!pip install -q -U accelerate

import numpy as np
import json, math, os, gc, re, time
from pathlib import Path
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/LRTIA')
BASE = DRIVE / 'Results/sentence_influence_matrix/llama'
BASE.mkdir(parents=True, exist_ok=True)
TARGETS_PATH = DRIVE / 'Results/corpus_expansion/targets_llama.jsonl'
TOK_MANIFEST_PATH = DRIVE / 'Results/corpus_expansion/tokenized_manifest_llama.jsonl'

MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B'

MAX_SENTENCES = 25            # cap per doc to keep compute bounded; opening is always sentence 0
MIN_SENTENCES = 12            # need enough sentences to have meaningful long-range pairs
DOCS_PER_CELL = 15            # pilot N
MIN_SENT_TOKENS = 3           # skip very short sentences (e.g. fragments)
SEED = 20260503

RUN_CORPORA = ['gutenberg_fiction_en', 'ted_transcripts_en']

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Probe: {MODEL_NAME}')
print(f'Per-doc: up to {MAX_SENTENCES} sentences, min {MIN_SENTENCES}')
print(f'Pilot: {DOCS_PER_CELL} docs/corpus × {len(RUN_CORPORA)} corpora')

In [ ]:
# Load model — fp16 for H100.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()
print(f'{MODEL_NAME} loaded (fp16)')

In [ ]:
# ---- ppl pipeline ----
@torch.no_grad()
def ppl_nll(ctx_toks, tgt_toks):
    if len(tgt_toks) < 2:
        return float('inf')
    full = list(ctx_toks) + list(tgt_toks)
    ts = len(ctx_toks)
    ids = torch.tensor([full], device=model.device)
    out = model(ids); logits = out.logits[0]
    nll = 0.0; cnt = 0
    for i in range(ts, len(full) - 1):
        lp = torch.log_softmax(logits[i], dim=-1)
        nll += -lp[full[i + 1]].item(); cnt += 1
    del out, logits; torch.cuda.empty_cache()
    if cnt == 0:
        return float('inf')
    return math.exp(nll / cnt)

# ---- multilingual sentence splitter ----
SENT_SPLIT_RE = re.compile(r'(?<=[.!?])\s+|(?<=[\u3002\uff01\uff1f])')
def split_sentences(text):
    parts = SENT_SPLIT_RE.split(text.strip())
    return [p.strip() for p in parts if len(p.strip()) >= 4]

# ---- compute the (n × n) influence matrix for one doc ----
def compute_influence_matrix(text):
    sents = split_sentences(text)
    if len(sents) < MIN_SENTENCES:
        return None
    if len(sents) > MAX_SENTENCES:
        sents = sents[:MAX_SENTENCES]
    sent_ids = [tokenizer.encode(s, add_special_tokens=False) for s in sents]
    # filter very short sentences (treat as not analyzable)
    valid = [i for i, s in enumerate(sent_ids) if len(s) >= MIN_SENT_TOKENS]
    if len(valid) < MIN_SENTENCES or 0 not in valid:
        return None
    n = len(sent_ids)

    # influence[i][t] = log(ppl_dropped_i / ppl_full)  for t > i
    matrix = [[None] * n for _ in range(n)]
    ppl_full_per_t = [None] * n

    for t in range(2, n):
        if len(sent_ids[t]) < MIN_SENT_TOKENS:
            continue
        full_ctx = [tok for k in range(t) for tok in sent_ids[k]]
        ppl_full = ppl_nll(full_ctx, sent_ids[t])
        if math.isinf(ppl_full) or ppl_full <= 0:
            continue
        ppl_full_per_t[t] = ppl_full
        for i in range(t):
            if len(sent_ids[i]) < MIN_SENT_TOKENS:
                continue
            ablated = [tok for k in range(t) if k != i for tok in sent_ids[k]]
            if len(ablated) < 2:
                continue
            ppl_drop = ppl_nll(ablated, sent_ids[t])
            if math.isinf(ppl_drop) or ppl_drop <= 0:
                continue
            matrix[i][t] = math.log(ppl_drop / ppl_full)

    return {
        'sentences': sents,
        'sentence_token_lengths': [len(s) for s in sent_ids],
        'influence_matrix': matrix,
        'ppl_full_per_target': ppl_full_per_t,
    }

print('Pipeline ready')

In [ ]:
# ---- document resolver ----
tok_manifest = {}
with open(TOK_MANIFEST_PATH) as f:
    for line in f:
        d = json.loads(line)
        tok_manifest[d['document_id']] = d['file_path']
ce_corpora = {}
with open(TARGETS_PATH) as f:
    for line in f:
        t = json.loads(line)
        ce_corpora.setdefault(t['corpus_id'], set()).add(t['document_id'])

def fix_ce_path(p):
    return str(p).replace('data/corpus_expansion/clean/',
        '/content/drive/MyDrive/LRTIA/Data/corpus_expansion/')

def get_doc_texts(corpus_id):
    for doc_id in sorted(ce_corpora.get(corpus_id, set())):
        fp = tok_manifest.get(doc_id)
        if fp is None: continue
        abs_fp = Path(fix_ce_path(fp))
        if not abs_fp.exists():
            abs_fp = Path('/content/drive/MyDrive/LRTIA/Data/corpus_expansion') / Path(fp).name
        try:
            yield doc_id, abs_fp.read_text(encoding='utf-8', errors='replace').strip()
        except Exception:
            continue

for c in RUN_CORPORA:
    print(f'  {c}: {sum(1 for _ in get_doc_texts(c))} total docs available')

In [ ]:
# ---- main run loop ----
for corpus_id in RUN_CORPORA:
    cache_path = BASE / f'{corpus_id}.json'
    if cache_path.exists():
        with open(cache_path) as f: n = len(json.load(f))
        print(f'\n{corpus_id}: cached ({n})'); continue

    print(f'\n{"="*60}\n{corpus_id}: target {DOCS_PER_CELL} docs\n{"="*60}')
    t0 = time.time()
    results = []
    skipped = 0

    pbar = tqdm(total=DOCS_PER_CELL, desc=corpus_id)
    for doc_id, text in get_doc_texts(corpus_id):
        if len(results) >= DOCS_PER_CELL:
            break
        out = compute_influence_matrix(text)
        if out is None:
            skipped += 1; continue
        out['corpus_id'] = corpus_id
        out['document_id'] = doc_id
        results.append(out)
        pbar.update(1)
    pbar.close()

    elapsed = time.time() - t0
    with open(cache_path, 'w') as f:
        json.dump(results, f)
    print(f'  {len(results)} docs in {elapsed/60:.1f} min ({skipped} skipped)')

    # quick spot-check: opening-vs-other influence at d=2 and d=10
    if results:
        for d in (2, 10):
            opn, oth = [], []
            for r in results:
                m = r['influence_matrix']
                n = len(m)
                for t in range(d, n):
                    v = m[t - d][t]
                    if v is None: continue
                    (opn if t - d == 0 else oth).append(v)
            if opn and oth:
                m_o = sum(opn) / len(opn); m_x = sum(oth) / len(oth)
                print(f'  d={d:>2}: opening mean={m_o:+.4f} (n={len(opn)})  '
                      f'others mean={m_x:+.4f} (n={len(oth)})  diff={m_o-m_x:+.4f}')

print('\nDone.')

In [ ]:
# ---- full analysis ----
from collections import defaultdict

def gini(values):
    v = sorted([x for x in values if x >= 0])
    n = len(v)
    if n == 0 or sum(v) == 0:
        return float('nan')
    s = sum((i + 1) * x for i, x in enumerate(v))
    return (2 * s) / (n * sum(v)) - (n + 1) / n

def analyze(corpus_id):
    fp = BASE / f'{corpus_id}.json'
    if not fp.exists():
        print(f'{corpus_id}: no cache'); return None
    with open(fp) as f:
        recs = json.load(f)
    if not recs:
        return None

    by_dist = defaultdict(list)
    by_dist_opening = defaultdict(list)
    by_dist_other = defaultdict(list)
    by_src_pos = defaultdict(list)

    for r in recs:
        m = r['influence_matrix']
        n = len(m)
        for t in range(2, n):
            for i in range(t):
                v = m[i][t]
                if v is None: continue
                d = t - i
                by_dist[d].append(v)
                if i == 0:
                    by_dist_opening[d].append(v)
                else:
                    by_dist_other[d].append(v)
                by_src_pos[i].append(v)

    print(f'\n{"="*70}\n{corpus_id}  ({len(recs)} docs)\n{"="*70}')

    print(f'\nDistance decay (all source positions pooled):')
    print(f'{"d":>4} {"n":>5} {"%pos":>7} {"mean":>9} {"median":>9}')
    for d in sorted(by_dist):
        vs = by_dist[d]
        if len(vs) < 5: continue
        pct = 100 * sum(1 for v in vs if v > 0) / len(vs)
        print(f'{d:>4} {len(vs):>5} {pct:>6.1f}% {sum(vs)/len(vs):>+9.4f} '
              f'{sorted(vs)[len(vs)//2]:>+9.4f}')

    print(f'\nOpening-position effect WITHIN distance bins:')
    print(f'{"d":>4} {"opening mean":>14} {"others mean":>13} {"diff":>8} '
          f'{"ratio":>7} {"n_open":>7}/{"n_oth":>5}')
    for d in sorted(set(by_dist_opening) | set(by_dist_other)):
        opn = by_dist_opening.get(d, [])
        oth = by_dist_other.get(d, [])
        if len(opn) < 3 or len(oth) < 5: continue
        mo = sum(opn) / len(opn)
        mx = sum(oth) / len(oth)
        ratio = (mo / mx) if abs(mx) > 1e-6 else float('inf')
        print(f'{d:>4} {mo:>+14.4f} {mx:>+13.4f} {mo-mx:>+8.4f} '
              f'{ratio:>7.2f} {len(opn):>7}/{len(oth):>5}')

    print(f'\nSource-position pooled (mean influence by source position, all distances):')
    print(f'{"src_pos":>8} {"n":>5} {"%pos":>7} {"mean":>9}')
    for sp in sorted(by_src_pos)[:15]:
        vs = by_src_pos[sp]
        if len(vs) < 5: continue
        print(f'{sp:>8} {len(vs):>5} {100*sum(1 for v in vs if v>0)/len(vs):>6.1f}% '
              f'{sum(vs)/len(vs):>+9.4f}')

    print(f'\nPer-doc concentration (% positive influence carried by top-k source sentences):')
    top1, top3 = [], []
    for r in recs:
        m = r['influence_matrix']; n = len(m)
        src_tot = []
        for i in range(n):
            s = 0
            for t in range(i + 1, n):
                v = m[i][t]
                if v is not None and v > 0:
                    s += v
            if s > 0: src_tot.append((i, s))
        if not src_tot: continue
        src_tot.sort(key=lambda x: -x[1])
        tot = sum(s for _, s in src_tot)
        top1.append((src_tot[0][0], src_tot[0][1] / tot * 100))
        if len(src_tot) >= 3:
            top3.append(sum(s for _, s in src_tot[:3]) / tot * 100)
    if top1:
        print(f'  median top-1 source share: {sorted(s for _, s in top1)[len(top1)//2]:.1f}%')
        print(f'  median top-3 source share: {sorted(top3)[len(top3)//2]:.1f}% (when n>=3)')
        # how often is the opening (src_pos=0) the top-1 contributor?
        n_open_top = sum(1 for pos, _ in top1 if pos == 0)
        print(f'  opening (src_pos=0) is the top-1 contributor in '
              f'{n_open_top}/{len(top1)} = {100*n_open_top/len(top1):.0f}% of docs')

    return {'recs': recs, 'by_dist': by_dist, 'by_dist_opening': by_dist_opening,
            'by_dist_other': by_dist_other, 'by_src_pos': by_src_pos}

results = {c: analyze(c) for c in RUN_CORPORA}

In [ ]:
# ---- diagnostic plots ----
import matplotlib.pyplot as plt

for corpus_id in RUN_CORPORA:
    r = results.get(corpus_id)
    if r is None: continue
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'{corpus_id} — Adult sentence-influence matrix  (n={len(r["recs"])} docs)',
                 fontsize=12, fontweight='bold')

    # 1. Distance decay (mean influence by distance)
    ax = axes[0, 0]
    ds = sorted([d for d in r['by_dist'] if len(r['by_dist'][d]) >= 5])
    means = [sum(r['by_dist'][d])/len(r['by_dist'][d]) for d in ds]
    pcts = [100*sum(1 for v in r['by_dist'][d] if v > 0)/len(r['by_dist'][d]) for d in ds]
    ax.plot(ds, means, 'o-', color='C0', linewidth=2, markersize=5, label='mean influence')
    ax2 = ax.twinx()
    ax2.plot(ds, pcts, 's--', color='C3', linewidth=1.5, markersize=4, alpha=0.7, label='% positive')
    ax2.set_ylabel('% positive', color='C3')
    ax2.set_ylim(0, 100)
    ax2.axhline(50, color='C3', linestyle=':', alpha=0.5)
    ax.set_xlabel('Sentence distance')
    ax.set_ylabel('Mean influence (log ppl ratio)', color='C0')
    ax.set_title('Distance decay')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.legend(loc='upper right')

    # 2. Opening vs other source positions, per distance — the key panel
    ax = axes[0, 1]
    ds_both = sorted(set(r['by_dist_opening']) & set(r['by_dist_other']))
    ds_both = [d for d in ds_both if len(r['by_dist_opening'][d]) >= 3 and len(r['by_dist_other'][d]) >= 5]
    opn_means = [sum(r['by_dist_opening'][d])/len(r['by_dist_opening'][d]) for d in ds_both]
    oth_means = [sum(r['by_dist_other'][d])/len(r['by_dist_other'][d]) for d in ds_both]
    ax.plot(ds_both, opn_means, 'o-', color='C2', linewidth=2.5, markersize=7,
            label='opening (src_pos=0)')
    ax.plot(ds_both, oth_means, 's-', color='C0', linewidth=2, markersize=5,
            label='other source positions')
    ax.fill_between(ds_both, opn_means, oth_means, where=[a > b for a, b in zip(opn_means, oth_means)],
                    color='C2', alpha=0.15, label='opening boost')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Sentence distance')
    ax.set_ylabel('Mean influence')
    ax.set_title('Opening anchor effect: opening vs others at SAME distance')
    ax.legend()

    # 3. Source-position effect (mean influence by source position)
    ax = axes[1, 0]
    sps = sorted([s for s in r['by_src_pos'] if len(r['by_src_pos'][s]) >= 5])[:25]
    sp_means = [sum(r['by_src_pos'][s])/len(r['by_src_pos'][s]) for s in sps]
    colors = ['C2' if s == 0 else 'C0' for s in sps]
    ax.bar(sps, sp_means, color=colors, alpha=0.8)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Source sentence position (0 = opening)')
    ax.set_ylabel('Mean influence (pooled across all distances)')
    ax.set_title('Source-position effect (green = opening)')

    # 4. Influence-matrix heatmap for the median-length doc
    ax = axes[1, 1]
    docs_by_n = sorted(r['recs'], key=lambda d: len(d['influence_matrix']))
    ex = docs_by_n[len(docs_by_n) // 2]
    n = len(ex['influence_matrix'])
    grid = np.full((n, n), np.nan)
    for i in range(n):
        for t in range(n):
            v = ex['influence_matrix'][i][t]
            if v is not None: grid[i, t] = v
    vmax = np.nanmax(np.abs(grid)) if not np.all(np.isnan(grid)) else 1
    im = ax.imshow(grid, cmap='RdBu_r', aspect='auto', vmin=-vmax, vmax=vmax)
    ax.set_xlabel('Target sentence (t)')
    ax.set_ylabel('Source sentence (i)  — opening at top')
    ax.set_title(f'Example matrix — doc {ex["document_id"][:25]}, n={n}')
    plt.colorbar(im, ax=ax, label='log(ppl_drop / ppl_full)', shrink=0.8)

    plt.tight_layout()
    out = BASE / f'{corpus_id}_anchor.png'
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'saved {out.name}')

In [ ]:
# ---- verdict ----
print('Reading the headline numbers:')
for corpus_id, r in results.items():
    if r is None: continue
    # Compute opening vs other ratio at long range (d >= 5).
    long_open = [v for d in r['by_dist_opening'] if d >= 5 for v in r['by_dist_opening'][d]]
    long_oth = [v for d in r['by_dist_other'] if d >= 5 for v in r['by_dist_other'][d]]
    if long_open and long_oth:
        m_o = sum(long_open) / len(long_open)
        m_x = sum(long_oth) / len(long_oth)
        ratio = m_o / m_x if abs(m_x) > 1e-6 else float('inf')
        print(f'\n  {corpus_id}:')
        print(f'    long-range (d>=5) opening mean: {m_o:+.4f}')
        print(f'    long-range (d>=5) other mean:  {m_x:+.4f}')
        print(f'    opening/other ratio (long-range): {ratio:.2f}×')
        if ratio >= 2.0:
            verdict = 'STRONG opening-anchor effect at long range — discourse-memory anchor model supported in adult prose'
        elif ratio >= 1.3:
            verdict = 'MODERATE opening-anchor effect — anchor model partially supported'
        else:
            verdict = 'NO opening-anchor effect at long range — pure positional/recency model holds in adult prose'
        print(f'    verdict: {verdict}')